<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Lecture 7: Evaluation in NLP

**University of Maryland, College Park**  
**Fall 2025**  
**Instructor**: Armin Mehrabian

## 🎯 Learning Objectives

By the end of this lecture, you will be able to:
1. Distinguish between different types of evaluation (intrinsic, extrinsic, situated, economic)
2. Understand formative vs. summative evaluation and their role in research
3. Calculate and interpret evaluation metrics: accuracy, precision, recall, F-measure
4. Implement cross-validation and understand data partitioning best practices
5. Compute inter-annotator agreement using Cohen's Kappa
6. Evaluate structured outputs (e.g., dependency trees) using precision and recall
7. Assess correlation between automatic metrics and human ratings

## 📋 Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import pearsonr

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Types of Evaluations

• **Intrinsic** - How well does the algorithm do at its task?

• **Extrinsic** - How well does it facilitate a larger task of which it is a component?

• **Situated** - How well does it perform in a real-use setting?

• **Economic** - What will people pay for it?

# Example - Google PageRank

### Intrinsic
- Controlled set of queries and documents and relevance assessments
- Formally defined evaluation measures - precision, recall, runtime, etc.

### Extrinsic
- External task, e.g. helping people answer a "factoid" question
- Formal measures for that external task, e.g. user time to answer questions
- Controlled experiment, but interaction with external factors situated
- In-use study (monitoring real searches with their questions)
  - Hard to conduct a controlled, replaceable experiment

# What Is an Economic Evaluation?

• People "vote with their pocketbooks"

• In addition to that, one can add subjective surveys, etc., instrumentation

• Many uncontrolled variables, sales/marketing manipulation, etc.

# Formative vs. Summative

• **Formative** informs algorithm designer about progress (fast, iterative)
  - When the cook tastes the soup

• **Summative** assesses whether goals are met at a major milestone
  - When the customer tastes the soup

• Historically, NLP / AI evaluations required a lot of manual effort

• Breakthrough comes with the implementation of automatic evaluations (BLEU, PARSEVAL metrics), which ushers a major change in nature of research
  - Also leads to automatic optimization to the evaluation criteria

In [ ]:
# Demonstrate perplexity calculation
def calculate_perplexity(probabilities):
    """
    Calculate perplexity from a sequence of probabilities
    probabilities: array of predicted probabilities for each token
    """
    # Perplexity = exp(cross-entropy) = exp(-1/N * sum(log(p_i)))
    log_probs = np.log(probabilities)
    cross_entropy = -np.mean(log_probs)
    perplexity = np.exp(cross_entropy)
    return perplexity, cross_entropy

# Example: Two language models predicting next token probabilities
np.random.seed(42)
sequence_length = 100

# Model A: Better model (higher probabilities)
probs_model_a = np.random.beta(8, 2, sequence_length)  # Skewed towards high probs

# Model B: Worse model (lower probabilities)
probs_model_b = np.random.beta(2, 5, sequence_length)  # Skewed towards low probs

perp_a, ce_a = calculate_perplexity(probs_model_a)
perp_b, ce_b = calculate_perplexity(probs_model_b)

print("Loss-Based Evaluation Example: Language Models")
print("="*60)
print(f"Model A:")
print(f"  Mean probability: {probs_model_a.mean():.3f}")
print(f"  Cross-entropy: {ce_a:.3f}")
print(f"  Perplexity: {perp_a:.3f}")
print(f"\nModel B:")
print(f"  Mean probability: {probs_model_b.mean():.3f}")
print(f"  Cross-entropy: {ce_b:.3f}")
print(f"  Perplexity: {perp_b:.3f}")
print(f"\n→ Lower perplexity = Better model")

# Visualize the probability distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(probs_model_a, bins=30, alpha=0.7, color='green', edgecolor='black')
ax1.axvline(probs_model_a.mean(), color='darkgreen', linestyle='--', linewidth=2, 
           label=f'Mean: {probs_model_a.mean():.3f}')
ax1.set_xlabel('Predicted Probability', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title(f'Model A: Perplexity = {perp_a:.2f}', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.hist(probs_model_b, bins=30, alpha=0.7, color='orange', edgecolor='black')
ax2.axvline(probs_model_b.mean(), color='darkorange', linestyle='--', linewidth=2,
           label=f'Mean: {probs_model_b.mean():.3f}')
ax2.set_xlabel('Predicted Probability', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title(f'Model B: Perplexity = {perp_b:.2f}', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Loss-Based Evaluation

• Modern ML systems often use **loss functions** as primary training signals
  - Examples: cross-entropy loss, perplexity

• **Perplexity** is a common evaluation metric for language models:

$$\text{Perplexity}(W) = P(w_1, w_2, \ldots, w_N)^{-\frac{1}{N}}$$

- Lower perplexity = better language model

### ⚠️ Important Caveat

• **Loss does not always correlate with task performance**
  - Example: Llama 2 training showed continued loss decrease but downstream accuracy plateaued
  - Validation loss ≠ capability on real-world tasks

• Always complement loss-based evaluation with task-specific metrics

# Shared Tasks (1/2)

• Emergence of "shared tasks"
  - (Kaggle, WMT, SemEval, etc.)

• A task is **always** a formalized abstraction of a problem

• Examples:
  - User who will not look past the first 10 "hits" (precision-at-10 as metric)
  - Finding one relevant document with needed information (time-biased gain)
  - User who interacts with system (interactive IR evaluations)

• **Takeaway**: Don't think in terms of "tasks." Think about questions and problems.

In [ ]:
# Example: Testing robustness to prompt variations
def evaluate_robustness(model_responses, prompt_variants):
    """
    Evaluate how consistent model performance is across prompt variations
    """
    # Simulate model responses to different prompt phrasings
    # In practice, you'd query an actual model
    
    print("Robustness Testing: Prompt Variation Analysis")
    print("="*60)
    print("\nTask: Answer 'What is 2+2?'\n")
    
    for i, (prompt, response) in enumerate(zip(prompt_variants, model_responses), 1):
        print(f"Variant {i}: '{prompt}'")
        print(f"  → Response: {response}")
        print()
    
    # Calculate consistency
    correct_responses = [r for r in model_responses if '4' in r]
    consistency = len(correct_responses) / len(model_responses)
    
    print(f"Consistency: {consistency:.1%} ({len(correct_responses)}/{len(model_responses)} variants correct)")
    
    return consistency

# Example with different prompt phrasings
prompts = [
    "What is 2+2?",
    "Calculate 2 plus 2",
    "2+2=",
    "If I have 2 apples and get 2 more, how many do I have?",
    "Solve: 2+2"
]

# Simulate model responses (robust model)
responses_robust = ["4", "4", "4", "4 apples", "4"]

# Simulate responses (less robust model) 
responses_fragile = ["4", "4", "The answer is 4", "You would have several apples", "2+2=4"]

print("Robust Model:")
consistency_robust = evaluate_robustness(responses_robust, prompts)

print("\n" + "="*60 + "\n")

print("Fragile Model:")
consistency_fragile = evaluate_robustness(responses_fragile, prompts)

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
models = ['Robust Model', 'Fragile Model']
consistencies = [consistency_robust, consistency_fragile]
colors = ['green', 'orange']

bars = ax.bar(models, consistencies, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Consistency Score', fontsize=12)
ax.set_title('Model Robustness to Prompt Variations', fontsize=14)
ax.set_ylim([0, 1.1])
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

# Add value labels on bars
for bar, val in zip(bars, consistencies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
           f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate pass@k calculation for code generation
from scipy.special import comb

def calculate_pass_at_k(n, c, k):
    """
    Calculate pass@k metric for code generation
    n: total number of samples generated
    c: number of samples that pass all tests
    k: number of samples to consider
    
    pass@k = E[1 - (n-c choose k) / (n choose k)]
         = probability that at least 1 of k samples is correct
    """
    if n - c < k:
        return 1.0
    return 1.0 - (comb(n - c, k) / comb(n, k))

# Example: Model generates 100 solutions, 40 are correct
n_samples = 100
n_correct = 40

print("pass@k Metric for Code Generation")
print("="*60)
print(f"Total samples generated: {n_samples}")
print(f"Correct solutions: {n_correct} ({n_correct/n_samples:.0%})")
print()

k_values = [1, 5, 10, 25, 50, 100]
pass_at_k_values = []

for k in k_values:
    if k <= n_samples:
        pass_k = calculate_pass_at_k(n_samples, n_correct, k)
        pass_at_k_values.append(pass_k)
        print(f"pass@{k:3d} = {pass_k:.3f} ({pass_k*100:.1f}%)")
    else:
        pass_at_k_values.append(None)

# Visualize
valid_k = [k for k, v in zip(k_values, pass_at_k_values) if v is not None]
valid_values = [v for v in pass_at_k_values if v is not None]

plt.figure(figsize=(10, 6))
plt.plot(valid_k, valid_values, 'o-', linewidth=2, markersize=8, color='darkblue')
plt.xlabel('k (number of samples)', fontsize=12)
plt.ylabel('pass@k', fontsize=12)
plt.title('pass@k vs. Number of Samples (40% correct)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1.05])
plt.axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Perfect score')
plt.axhline(y=n_correct/n_samples, color='red', linestyle='--', alpha=0.5, label='Accuracy (40%)')
plt.legend()
plt.tight_layout()
plt.show()

print("\n→ pass@k increases with k because we have more chances to get a correct solution")

## 3. GSM8k (Grade School Math)

• Tests **mathematical reasoning** with word problems
• Format: Natural language math problem → numerical answer
• Evaluates: Multi-step reasoning, arithmetic
• Metric: Exact match accuracy

**Example Problem:**
```
Question: Janet's ducks lay 16 eggs per day. She eats three for 
         breakfast every morning and bakes muffins for her friends 
         every day with four. She sells the remainder at the farmers' 
         market daily for $2 per fresh duck egg. How much does she 
         make every day at the farmers' market?

Solution: 
Step 1: Eggs laid per day = 16
Step 2: Eggs eaten = 3
Step 3: Eggs used for baking = 4
Step 4: Eggs remaining = 16 - 3 - 4 = 9
Step 5: Revenue = 9 × $2 = $18

Answer: $18
```

### Chain-of-Thought Evaluation
• Models often show reasoning steps before final answer
• Can evaluate both final answer accuracy AND reasoning quality

## 2. HumanEval (Code Generation)

• Tests ability to generate **functional Python code**
• Format: Function signature + docstring → complete implementation
• Evaluates: Programming capability, algorithmic thinking
• Metric: **pass@k** (probability that at least 1 of k generated solutions passes tests)

**Example Problem:**
```python
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer 
        to each other than given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    # Model must generate implementation here
```

### Understanding pass@k

• **pass@1**: Does the first generated solution work?
• **pass@10**: Do any of 10 generated solutions work?
• Higher k accounts for model stochasticity and multiple approaches

# Modern Benchmark Examples

Let's examine three influential benchmarks that exemplify different evaluation paradigms:

## 1. MMLU (Massive Multitask Language Understanding)

• Tests knowledge across **57 subjects** from STEM to humanities
• Format: Multiple-choice questions (4 options)
• Evaluates: Factual knowledge, reasoning, comprehension
• Metric: Accuracy (% correct)

**Example Question:**
```
Subject: High School Physics
Question: A car accelerates uniformly from rest. After 10 seconds, 
         its velocity is 20 m/s. What is its acceleration?
A) 0.5 m/s²
B) 2 m/s²
C) 10 m/s²
D) 200 m/s²

Answer: B
```

### 5. **Data Contamination**
- **Critical issue**: Test data may have appeared in training data
- LLMs trained on web-scale data may have "seen" benchmark problems
- Mitigation strategies:
  - Use newly created benchmarks post-training cutoff
  - Check for memorization with canary strings
  - Report training data overlap when known

### 6. **Efficient Evaluation**
- Fast, automated evaluation when possible
- Balance between thoroughness and computational cost
- Enable rapid iteration during development

# Properties of Good Benchmarks

What makes a benchmark truly effective? Consider these key properties:

### 1. **Breadth and Diversity**
- Coverage across multiple domains, tasks, and scenarios
- Example: MMLU covers 57 subjects from STEM to humanities
- Avoids overfitting to narrow distributions

### 2. **Depth and Difficulty**
- Tasks should be challenging and not quickly saturate
- Monitor benchmark saturation over time
- Include problems that require reasoning, not just memorization

### 3. **Utility**
- Measures capabilities relevant to real-world applications
- Aligns with downstream task requirements
- Predictive of practical performance

### 4. **Robustness**
- Resistant to minor prompt variations
- Stable across different phrasings
- Not overly sensitive to formatting

# Shared Tasks (2/2)

```
                        Shared tasks
                             |
         Role of task organizer
                             |
       +---------------------+----------------------+
       |                     |                      |
  "Task"                Training data             Test
  definition                 |                      |
                             v                      v
                          systems   ----------->   Eval
                                                    |
                                                    v
                                            Compare results
                                                    |
                                            -----------------
                                            -----------------
                                            -----------------
                                                    |
                                        "Performance" numbers
                                                    |
                                          Discussions / Insight
                                                    |
                                                Iteration
```

# The "Leaderboard" Phenomenon and "SOTA-chasing"

• Hyper-attention to "tasks", not problems or research questions

• Overfitting to specific datasets and formations

• Rank on leaderboard is not always equal to value or utility

• Incentive to focus only on innovation that steps a system up in the rankings using its single "figure of merit"

• Neglect of prediction cost (e.g. model size, training time, inference latency)

• Lack of robustness

# How Evaluation Guides AI Research

• Cohen and Howe (AI magazine 1988) argue for evaluation to be integrated throughout the research cycle.

1. Is the task significant? Why? Validate your formulation of the task.

2. Is your research likely to meaningfully contribute to the problem? Is the task tractable?

3. As the task becomes specifically defined for your research, is it still representative of a class of tasks?

4. Have any interesting aspects been abstracted away or simplified?

5. How will you know when you have succeeded in demonstrating a solution? Alternatively, is there a task where a solution can be demonstrated?

# Evaluation Comparison: Elements of a "Results" Table

| System                            | Description                                  |
|-----------------------------------|----------------------------------------------|
| Baseline                          | lower bound, typically a straightforward approach |
| Previous A<br>Previous B          | previous literature, existing state of the art |
| Ours #1<br>Ours #2                | new approach(es)                             |
| Oracle or inter-annotator agreement | upper bound                               |

# Absolute vs. Relative Improvement vs. Error Reduction

**Baseline:** 80%  
**Ours:** 85%

• **Absolute improvement**: 85% - 80% = 5%

• **Relative improvement**: (85% - 80%) / 80% = 6.25%

• **Relative reduction in error rate**: (20% - 15%) / 20% = 25%

• **Notice**: 98% to 99% is only 1% absolute, which seems unimpressive – but this is a full 50% reduction in the error rate

In [ ]:
# Demonstrate different improvement metrics
def calculate_improvements(baseline, improved):
    """Calculate absolute, relative, and error reduction metrics"""
    absolute_improvement = improved - baseline
    relative_improvement = (improved - baseline) / baseline * 100
    
    baseline_error = 1 - baseline
    improved_error = 1 - improved
    error_reduction = (baseline_error - improved_error) / baseline_error * 100
    
    print(f"Baseline: {baseline*100:.1f}%")
    print(f"Improved: {improved*100:.1f}%")
    print(f"\nAbsolute improvement: {absolute_improvement*100:.1f} percentage points")
    print(f"Relative improvement: {relative_improvement:.2f}%")
    print(f"Error reduction: {error_reduction:.2f}%")
    print()

# Example 1: 80% to 85%
print("Example 1: 80% to 85%")
print("="*50)
calculate_improvements(0.80, 0.85)

# Example 2: 98% to 99%
print("\nExample 2: 98% to 99%")
print("="*50)
calculate_improvements(0.98, 0.99)

# Partitioning of Data for Evaluation

• E.g. 70% for training, 20% for dev/tuning, 5% for devtest (formative), 5% for test (summative)

• **Cardinal rule**: don't test on training data if you want predictive value

• Beware "leakage" of training data into test data

• Overfitting underestimates true error rate

# Cross-validation

• Provides for intervals rather than point estimates of performance (cf. bootstrapping)

• Uses all the data while still properly maintaining train/test split

```
        Test                               
        ┌───┐        ┌───┐                 ┌───┐
        │   │        │   │     . . . .     │   │
        │   │        │   │                 │   │
        │   │        │   │                 │   │
        └───┘        └───┘                 └───┘
  i =    1            2                     k          "k-fold"
        m₁           m₂                    mₖ          mᵢ = value of evaluation measure on this fold
        └─────────────────────────────────┘
          mean, stdev, confidence intervals, etc.
```

• **Stratified cross-validation** - preserves proportions of labels in each split

In [ ]:
# Demonstrate cross-validation
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# Generate synthetic classification data
X, y = make_classification(n_samples=200, n_features=20, n_informative=15, 
                          n_redundant=5, random_state=42)

# Create a simple classifier
clf = LogisticRegression(max_iter=1000, random_state=42)

# Perform 10-fold cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

print("10-Fold Cross-Validation Results")
print("="*50)
print(f"Fold scores: {[f'{s:.3f}' for s in scores]}")
print(f"\nMean accuracy: {scores.mean():.3f}")
print(f"Standard deviation: {scores.std():.3f}")
print(f"95% Confidence interval: [{scores.mean() - 1.96*scores.std():.3f}, {scores.mean() + 1.96*scores.std():.3f}]")

# Visualize the scores
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), scores, 'o-', linewidth=2, markersize=8)
plt.axhline(y=scores.mean(), color='r', linestyle='--', label=f'Mean: {scores.mean():.3f}')
plt.fill_between(range(1, 11), scores.mean() - scores.std(), scores.mean() + scores.std(), 
                alpha=0.2, color='r', label=f'±1 Std Dev')
plt.xlabel('Fold', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('10-Fold Cross-Validation Scores', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Inter-annotator Agreement and Upper Limits

• Cf. Chang *et al.*: "agreement" measures goodness

• Gale *et al.* (1992): human agreement can act as an upper bound on performance

• **Accuracy** = # Agreements / # Items

• But there is also some agreement by chance….

### Chance-corrected agreement

$$K = \frac{A_0 - A_e}{1 - A_e}$$

where:  
- $A_0$ = observed agreement
- $A_e$ = estimate of chance agreement

**Definitive reference**: Artstein, R., & Poesio, M. (2008). Inter-coder agreement for computational linguistics. *Computational Linguistics*, 34(4), 555-596.

# Example Calculation of Cohen's Kappa

$$K = \frac{A_0 - A_e}{1 - A_e}$$

where:  
- $A_o$ = observed agreement
- $A_e$ = agreement by chance (if $A_e$ = 0, denominator equals $A_o$)

### Confusion Matrix: Rater 1 vs Rater 2

|            | psychotic | borderline | none | Total |
|------------|-----------|------------|------|-------|
| psychotic  | 10        | 4          | 1    | 15    |
| borderline | 6         | 16         | 2    | 24    |
| none       | 0         | 3          | 8    | 11    |
| **Total**  | 16        | 23         | 11   | 50    |

Diagonal counts = observed agreements

### Observed agreement:

$$A_0 = \frac{10 + 16 + 8}{50} = \frac{34}{50} = 0.68 = 68\%$$

### Chance agreement assuming independence of annotation:

$$A_e = \sum_{k \in \{\text{psychotic, borderline, none}\}} \left(\frac{N_{k,1}}{N} \times \frac{N_{k,2}}{N}\right) = \frac{1}{N^2}\sum_k N_{k,1}N_{k,2}$$

$$N_{\text{psychotic, rater1}} \cdot N_{\text{psychotic, rater2}} = \frac{16}{50} \times \frac{15}{50} = .32 \times .3 = .096$$

Computation left as an exercise…

**K = 0.496**

In [ ]:
# Implement Cohen's Kappa calculation
def cohens_kappa(confusion_matrix):
    """Calculate Cohen's Kappa from a confusion matrix"""
    n = confusion_matrix.sum()
    
    # Observed agreement (sum of diagonal)
    observed_agreement = np.trace(confusion_matrix) / n
    
    # Expected agreement by chance
    row_sums = confusion_matrix.sum(axis=1)
    col_sums = confusion_matrix.sum(axis=0)
    expected_agreement = np.sum(row_sums * col_sums) / (n ** 2)
    
    # Cohen's Kappa
    if expected_agreement == 1:
        return 1.0  # Perfect agreement
    kappa = (observed_agreement - expected_agreement) / (1 - expected_agreement)
    
    return kappa, observed_agreement, expected_agreement

# Example from slide: psychotic, borderline, none
cm_example1 = np.array([
    [10, 4, 1],
    [6, 16, 2],
    [0, 3, 8]
])

kappa, obs_agree, exp_agree = cohens_kappa(cm_example1)

print("Example 1: Psychiatric Diagnosis")
print("="*50)
print("Confusion Matrix:")
print(cm_example1)
print(f"\nObserved agreement (A₀): {obs_agree:.3f} = {obs_agree*100:.1f}%")
print(f"Expected agreement (Aₑ): {exp_agree:.3f} = {exp_agree*100:.1f}%")
print(f"Cohen's Kappa (K): {kappa:.3f}")

# Visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_example1, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['psychotic', 'borderline', 'none'],
           yticklabels=['psychotic', 'borderline', 'none'],
           cbar_kws={'label': 'Count'})
plt.xlabel('Rater 2', fontsize=12)
plt.ylabel('Rater 1', fontsize=12)
plt.title(f'Inter-Rater Agreement (κ = {kappa:.3f})', fontsize=14)
plt.tight_layout()
plt.show()

# Another Example Calculation of Cohen's Kappa

### Confusion Matrix: J1 vs J2

|     | k=1 | k=2 | Total |
|-----|-----|-----|-------|
| k=1 | 14  | 2   | 16    |
| k=2 | 2   | 22  | 24    |
| Total | 16 | 24 | N=40  |

### Observed agreement:

$$A_0 = \frac{14 + 22}{40} = \frac{36}{40} = \frac{9}{10} = 0.9$$

### Chance agreement:

$$A_{k=1} = \left(\frac{16}{40}\right)\left(\frac{16}{40}\right)$$

$$A_{k=2} = \left(\frac{24}{40}\right)\left(\frac{24}{40}\right)$$

$$\Rightarrow A_e = \frac{16^2}{40^2} + \frac{24^2}{40^2} = \frac{256 + 576}{40^2} = \frac{832}{1600} = 0.52$$

### Cohen's Kappa:

$$K = \frac{A_0 - A_e}{1 - A_e} = \frac{.9 - .52}{.48} = \frac{.38}{.48} = .792$$

Chance-corrected agreement lower than $A_0$ = .9 because of 0.52 probability of chance agreement

In [ ]:
# Example 2 from slide
cm_example2 = np.array([
    [14, 2],
    [2, 22]
])

kappa2, obs_agree2, exp_agree2 = cohens_kappa(cm_example2)

print("Example 2: Binary Classification")
print("="*50)
print("Confusion Matrix:")
print(cm_example2)
print(f"\nObserved agreement (A₀): {obs_agree2:.3f} = {obs_agree2*100:.1f}%")
print(f"Expected agreement (Aₑ): {exp_agree2:.3f} = {exp_agree2*100:.1f}%")
print(f"Cohen's Kappa (K): {kappa2:.3f}")

# Visualize
plt.figure(figsize=(6, 5))
sns.heatmap(cm_example2, annot=True, fmt='d', cmap='Greens',
           xticklabels=['k=1', 'k=2'],
           yticklabels=['k=1', 'k=2'],
           cbar_kws={'label': 'Count'})
plt.xlabel('Judge 2 (J2)', fontsize=12)
plt.ylabel('Judge 1 (J1)', fontsize=12)
plt.title(f'Inter-Rater Agreement (κ = {kappa2:.3f})', fontsize=14)
plt.tight_layout()
plt.show()

## Human Evaluation: The Gold Standard

Despite automation advances, human evaluation remains essential:

### Traditional Methods
- **Likert scales** (1-5 or 1-7): Rate quality dimensions
- **Ranking**: Order multiple outputs by preference
- **Binary comparison**: Which output is better?

### Modern Platforms: Chatbot Arena

• Users interact with **anonymous chatbot pairs**
• After conversation, vote for which was better
• Aggregate thousands of comparisons
• Compute **Elo ratings** (like chess rankings)

**Why it works:**
- Real user interactions (not contrived test cases)
- Blind evaluation (no brand bias)
- Large scale (crowdsourced)
- Reflects actual user preferences

**Limitations:**
- Still expensive and slow at scale
- Can have demographic biases in raters
- Preferences may not align with specific use cases

## LLM-as-Judge

Modern approach: Use a **strong LLM** (e.g., GPT-4) to evaluate generated text

### How it works:
1. Provide evaluation criteria in a prompt
2. Show the LLM the generated text
3. LLM outputs a score or judgment

**Example Prompt:**
```
Rate the following summary on a scale of 1-5 for:
- Faithfulness (does it accurately reflect the source?)
- Coverage (does it capture key points?)
- Fluency (is it well-written?)

Source: [original text]
Summary: [generated summary]
```

### Pros & Cons

✅ Can evaluate nuanced qualities (coherence, style, factuality)  
✅ No need for reference texts  
✅ Flexible evaluation criteria  
❌ Expensive (API costs)  
❌ Slower than metrics like ROUGE  
❌ Can have biases (e.g., prefers longer outputs, own style)

In [ ]:
# Implement simplified ROUGE-N calculation
def calculate_rouge_n(generated, reference, n=1):
    """
    Calculate ROUGE-N score (simplified version)
    generated: generated text string
    reference: reference text string
    n: n-gram size (1 for unigrams, 2 for bigrams, etc.)
    """
    def get_ngrams(text, n):
        """Extract n-grams from text"""
        words = text.lower().split()
        ngrams = []
        for i in range(len(words) - n + 1):
            ngrams.append(tuple(words[i:i+n]))
        return ngrams
    
    gen_ngrams = get_ngrams(generated, n)
    ref_ngrams = get_ngrams(reference, n)
    
    if len(ref_ngrams) == 0:
        return 0.0
    
    # Count matches
    ref_ngram_counts = {}
    for ngram in ref_ngrams:
        ref_ngram_counts[ngram] = ref_ngram_counts.get(ngram, 0) + 1
    
    matches = 0
    for ngram in gen_ngrams:
        if ngram in ref_ngram_counts and ref_ngram_counts[ngram] > 0:
            matches += 1
            ref_ngram_counts[ngram] -= 1
    
    # ROUGE is recall-oriented: matches / reference_count
    recall = matches / len(ref_ngrams)
    
    # Also calculate precision and F1
    precision = matches / len(gen_ngrams) if len(gen_ngrams) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return recall, precision, f1

# Example: Summarization
reference = "The quick brown fox jumps over the lazy dog"
generated_good = "The brown fox jumps over the dog"
generated_poor = "A cat sleeps on the mat"

print("ROUGE Evaluation Example: Text Summarization")
print("="*60)
print(f"Reference:  '{reference}'")
print(f"Generated (good): '{generated_good}'")
print(f"Generated (poor): '{generated_poor}'")
print()

for n in [1, 2]:
    print(f"\nROUGE-{n}:")
    print("-" * 60)
    
    recall_good, prec_good, f1_good = calculate_rouge_n(generated_good, reference, n)
    recall_poor, prec_poor, f1_poor = calculate_rouge_n(generated_poor, reference, n)
    
    print(f"Good summary: Recall={recall_good:.3f}, Precision={prec_good:.3f}, F1={f1_good:.3f}")
    print(f"Poor summary: Recall={recall_poor:.3f}, Precision={prec_poor:.3f}, F1={f1_poor:.3f}")

# Visualize comparison
metrics = ['ROUGE-1\nRecall', 'ROUGE-1\nF1', 'ROUGE-2\nRecall', 'ROUGE-2\nF1']
good_scores = [
    calculate_rouge_n(generated_good, reference, 1)[0],
    calculate_rouge_n(generated_good, reference, 1)[2],
    calculate_rouge_n(generated_good, reference, 2)[0],
    calculate_rouge_n(generated_good, reference, 2)[2]
]
poor_scores = [
    calculate_rouge_n(generated_poor, reference, 1)[0],
    calculate_rouge_n(generated_poor, reference, 1)[2],
    calculate_rouge_n(generated_poor, reference, 2)[0],
    calculate_rouge_n(generated_poor, reference, 2)[2]
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, good_scores, width, label='Good Summary', color='green', alpha=0.7)
bars2 = ax.bar(x + width/2, poor_scores, width, label='Poor Summary', color='red', alpha=0.7)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('ROUGE Scores: Good vs Poor Summary', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Variance Reduction Techniques

Goal: Get more reliable estimates with the same amount of data

### 1. Stratified Sampling
• Ensure test set has balanced representation of:
  - Different difficulty levels
  - Different categories/labels
  - Different data sources
• Reduces variance compared to random sampling

### 2. Clustered Standard Errors
• **Problem**: Test examples may not be independent
  - Multiple questions from same source document
  - Multiple turns in same conversation
  - Multiple examples from same user

• **Solution**: Account for clustering in statistical tests
  - Cluster by document, conversation, user, etc.
  - Use clustered standard errors (wider CIs)

### 3. Control Variates
• Use a correlated "control" metric to reduce variance
• Example: When evaluating on new dataset, also evaluate baseline to account for dataset difficulty

**Key insight**: Always think about data dependencies and account for them in your statistics!

In [ ]:
# Demonstrate paired vs. unpaired testing
from scipy.stats import ttest_ind, ttest_rel

# Generate example data: per-example F1 scores for two models on same test set
np.random.seed(42)
n_examples = 100

# Model A scores
model_a_scores = np.random.beta(8, 2, n_examples)

# Model B scores (correlated with A because same examples)
# Add noise but maintain correlation
model_b_scores = model_a_scores + np.random.normal(-0.05, 0.15, n_examples)
model_b_scores = np.clip(model_b_scores, 0, 1)

# Perform paired t-test
t_paired, p_paired = ttest_rel(model_a_scores, model_b_scores)

# Perform unpaired t-test (incorrect for this scenario!)
t_unpaired, p_unpaired = ttest_ind(model_a_scores, model_b_scores)

print("Paired vs. Unpaired Statistical Testing")
print("="*60)
print(f"Model A mean score: {model_a_scores.mean():.3f} ± {model_a_scores.std():.3f}")
print(f"Model B mean score: {model_b_scores.mean():.3f} ± {model_b_scores.std():.3f}")
print(f"Mean difference: {(model_a_scores - model_b_scores).mean():.3f}")
print()
print("Paired t-test (CORRECT - same examples):")
print(f"  t-statistic: {t_paired:.3f}")
print(f"  p-value: {p_paired:.4f}")
print(f"  Result: {'Significant' if p_paired < 0.05 else 'Not significant'} at α=0.05")
print()
print("Unpaired t-test (INCORRECT for this scenario):")
print(f"  t-statistic: {t_unpaired:.3f}")
print(f"  p-value: {p_unpaired:.4f}")
print(f"  Result: {'Significant' if p_unpaired < 0.05 else 'Not significant'} at α=0.05")
print()
print("→ Paired test is more powerful because it accounts for correlation")

# Visualize the pairing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot showing pairing
ax1.scatter(model_a_scores, model_b_scores, alpha=0.5, s=30)
ax1.plot([0, 1], [0, 1], 'r--', linewidth=2, label='y=x (no difference)')
ax1.set_xlabel('Model A Score', fontsize=12)
ax1.set_ylabel('Model B Score', fontsize=12)
ax1.set_title('Paired Scores (each point = one example)', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])

# Difference distribution
differences = model_a_scores - model_b_scores
ax2.hist(differences, bins=30, alpha=0.7, color='purple', edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='No difference')
ax2.axvline(differences.mean(), color='darkgreen', linestyle='-', linewidth=2, 
           label=f'Mean diff: {differences.mean():.3f}')
ax2.set_xlabel('Score Difference (A - B)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Paired Differences', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Paired vs. Unpaired Testing

When comparing two models, the statistical test depends on whether evaluations are **paired**.

### Paired Testing
• **When**: Same test set, same examples for both models
• **Why paired**: Controls for example difficulty variation
• **Test**: Paired t-test, McNemar's test (for binary outcomes)
• **More powerful**: Reduces variance, easier to detect real differences

### Unpaired Testing
• **When**: Different test sets or independent samples
• **Why unpaired**: Different evaluation conditions
• **Test**: Independent t-test, Mann-Whitney U test
• **Less powerful**: More variance, harder to detect differences

### Rule of thumb:
**Always use paired testing when possible** – it's more statistically efficient!

In [ ]:
# Demonstrate bootstrap confidence intervals
def bootstrap_ci(y_true, y_pred, metric_fn, n_bootstrap=10000, confidence=0.95):
    """
    Calculate bootstrap confidence interval for a metric
    """
    np.random.seed(42)
    scores = []
    n = len(y_true)
    
    for _ in range(n_bootstrap):
        # Resample with replacement
        indices = np.random.choice(n, size=n, replace=True)
        y_true_sample = y_true[indices]
        y_pred_sample = y_pred[indices]
        
        # Calculate metric on this resample
        score = metric_fn(y_true_sample, y_pred_sample)
        scores.append(score)
    
    scores = np.array(scores)
    
    # Calculate confidence interval
    alpha = 1 - confidence
    lower_percentile = (alpha / 2) * 100
    upper_percentile = (1 - alpha / 2) * 100
    
    ci_lower = np.percentile(scores, lower_percentile)
    ci_upper = np.percentile(scores, upper_percentile)
    point_estimate = metric_fn(y_true, y_pred)
    
    return point_estimate, ci_lower, ci_upper, scores

# Example: Two models on classification task
np.random.seed(42)
n_samples = 200

# Generate test data
y_true = np.random.randint(0, 2, n_samples)

# Model A: 85% accuracy
y_pred_a = y_true.copy()
n_errors_a = int(0.15 * n_samples)
error_indices_a = np.random.choice(n_samples, n_errors_a, replace=False)
y_pred_a[error_indices_a] = 1 - y_pred_a[error_indices_a]

# Model B: 82% accuracy
y_pred_b = y_true.copy()
n_errors_b = int(0.18 * n_samples)
error_indices_b = np.random.choice(n_samples, n_errors_b, replace=False)
y_pred_b[error_indices_b] = 1 - y_pred_b[error_indices_b]

# Calculate bootstrap CIs
from sklearn.metrics import accuracy_score

est_a, ci_lower_a, ci_upper_a, scores_a = bootstrap_ci(y_true, y_pred_a, accuracy_score)
est_b, ci_lower_b, ci_upper_b, scores_b = bootstrap_ci(y_true, y_pred_b, accuracy_score)

print("Bootstrap Confidence Intervals")
print("="*60)
print(f"Model A: {est_a:.3f} [95% CI: {ci_lower_a:.3f}, {ci_upper_a:.3f}]")
print(f"Model B: {est_b:.3f} [95% CI: {ci_lower_b:.3f}, {ci_upper_b:.3f}]")
print()
print("→ Confidence intervals overlap, suggesting difference may not be significant")

# Visualize bootstrap distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model A
axes[0].hist(scores_a, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[0].axvline(est_a, color='darkgreen', linestyle='-', linewidth=2, label=f'Point Est: {est_a:.3f}')
axes[0].axvline(ci_lower_a, color='red', linestyle='--', linewidth=2, label=f'95% CI')
axes[0].axvline(ci_upper_a, color='red', linestyle='--', linewidth=2)
axes[0].fill_betweenx([0, axes[0].get_ylim()[1]], ci_lower_a, ci_upper_a, alpha=0.2, color='red')
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Model A: Bootstrap Distribution', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Model B
axes[1].hist(scores_b, bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[1].axvline(est_b, color='darkblue', linestyle='-', linewidth=2, label=f'Point Est: {est_b:.3f}')
axes[1].axvline(ci_lower_b, color='red', linestyle='--', linewidth=2, label=f'95% CI')
axes[1].axvline(ci_upper_b, color='red', linestyle='--', linewidth=2)
axes[1].fill_betweenx([0, axes[1].get_ylim()[1]], ci_lower_b, ci_upper_b, alpha=0.2, color='red')
axes[1].set_xlabel('Accuracy', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Model B: Bootstrap Distribution', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical Rigor in Evaluation

Beyond point estimates, we need to quantify **uncertainty** and perform **statistical testing**.

## Confidence Intervals

A point estimate (e.g., "accuracy = 0.85") doesn't tell the full story. We need intervals!

### Bootstrap Confidence Intervals

1. Resample test set with replacement many times (e.g., 10,000)
2. Calculate metric on each resample
3. Report percentiles (e.g., 2.5th and 97.5th for 95% CI)

**Example**: If Model A achieves 85% accuracy, the 95% CI might be [82%, 88%]

### Why it matters:
- Small test sets → wide confidence intervals → less certainty
- Overlapping CIs → differences may not be significant
- Guides decision-making about model deployment

# Automatic Evaluation of Generated Text

Evaluating open-ended text generation (summaries, translations, dialogue) is challenging:
- No single "correct" answer
- Human evaluation is expensive and slow
- Need automated proxies for quality

## ROUGE: Recall-Oriented Understudy for Gisting Evaluation

Originally designed for summarization, ROUGE measures **n-gram overlap** between generated and reference texts.

### Key ROUGE Variants

• **ROUGE-N**: N-gram overlap (ROUGE-1 = unigrams, ROUGE-2 = bigrams)

$$\text{ROUGE-N} = \frac{\sum_{S \in \text{Refs}} \sum_{\text{gram}_n \in S} \text{Count}_{\text{match}}(\text{gram}_n)}{\sum_{S \in \text{Refs}} \sum_{\text{gram}_n \in S} \text{Count}(\text{gram}_n)}$$

• **ROUGE-L**: Longest Common Subsequence (measures sentence-level structure)

• **ROUGE-W**: Weighted LCS (favors consecutive matches)

### Strengths & Limitations

✅ Fast, automatic, reproducible  
✅ Correlates reasonably with human judgments  
❌ Only captures surface overlap, not meaning  
❌ Can be "gamed" with n-gram copying  
❌ Requires reference texts

# Evaluation with One Output Per Input, Exact Match

• Ex: Part-of-speech tagging

• **Accuracy**: number correct / number of items

• Variations:
  - Breakdown by category
  - Accuracy on just ambiguous cases (helps avoid inflated performance)
  - Allowing system to abstain from predictions:
    - **Accuracy** = # correct / # attempted
    - **Coverage** = # attempted / # items
    - **Recall** = # correct / # items
    - (Definitions vary, read carefully!)
  - Confusion matrix

# Evaluation with Multiple Responses Per Item

• Ex: Document retrieval

```
        docs
         ┌───┐  ✓
System   │   │ ✓
(=retrieved)
         ┌───┐  ✓
       ✓ │   │ ✓   Judge/Assessor
         └───┘      (="truth")
                    (="relevant")
```

**Precision** = |retrieved and relevant| / |retrieved| = 2/3

**Recall** = |retrieved and relevant| / |relevant| = 2/5

### Consider extrema:
• Retrieve all documents: 100% recall!
• Retrieve no documents: max precision (no false positives!)

# Evaluation with Multiple Responses Per Item

### Confusion Matrix

|            | Truth +            | Truth -                 |
|------------|--------------------|-----------------------|
| **Prediction +** | TP (True Positive) | FP (False Positive)<br>Type I error |
| **Prediction -** | FN (False Negative)<br>"Miss" Type II error | TN (True Negative) |

```
                    ┌──────────────────────────┐
                    │         TN               │
                    │   ┌────────────┐         │
                    │   │ FP    TP   │ FN      │  ← Actually positive
                    │   └────────────┘         │
                    └──────────────────────────┘
                           ↑
                    Predicted positive
```

- **Predicted positives** = TP + FP
- **Actual positives** = TP + FN

In [ ]:
# Demonstrate precision and recall calculations
def calculate_metrics(y_true, y_pred):
    """Calculate precision, recall, F1 from predictions"""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return tp, fp, fn, tn, precision, recall, f1

# Example: Document retrieval
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])  # 5 relevant documents
y_pred = np.array([1, 1, 0, 0, 0, 1, 0, 0, 0, 0])  # Retrieved 3 documents

tp, fp, fn, tn, precision, recall, f1 = calculate_metrics(y_true, y_pred)

print("Document Retrieval Example")
print("="*50)
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Negatives (TN): {tn}")
print(f"\nPrecision: {precision:.3f} = {tp}/{tp+fp}")
print(f"Recall: {recall:.3f} = {tp}/{tp+fn}")
print(f"F1 Score: {f1:.3f}")

# Visualize confusion matrix
cm = np.array([[tn, fp], [fn, tp]])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Predicted Negative', 'Predicted Positive'],
           yticklabels=['Actual Negative', 'Actual Positive'],
           cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Combining Precision and Recall Into a Single Figure of Merit: F-measure

$$F = \frac{1}{\alpha \frac{1}{P} + (1-\alpha)\frac{1}{R}}$$

weighted harmonic mean

$$\frac{2PR}{P+R} \quad \text{where} \quad \alpha = \frac{1}{2}$$

common case, equally weighting precision and recall

• Provides a single figure of merit for comparison/ranking

• Penalizes very different P/R values (hence discourages trading one off against the other)

• Biased toward maximizing true estimates, compared to accuracy's sensitivity to only the number of classification errors

In [ ]:
# Visualize the trade-off between precision and recall
def plot_precision_recall_tradeoff():
    """Plot F1 scores for different precision/recall combinations"""
    precisions = np.linspace(0.1, 1.0, 50)
    recalls = np.linspace(0.1, 1.0, 50)
    P, R = np.meshgrid(precisions, recalls)
    F1 = 2 * P * R / (P + R)
    
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(P, R, F1, levels=20, cmap='viridis')
    plt.colorbar(contour, label='F1 Score')
    
    # Add contour lines
    contour_lines = plt.contour(P, R, F1, levels=10, colors='white', alpha=0.3, linewidths=0.5)
    plt.clabel(contour_lines, inline=True, fontsize=8)
    
    plt.xlabel('Precision', fontsize=12)
    plt.ylabel('Recall', fontsize=12)
    plt.title('F1 Score as a Function of Precision and Recall', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_precision_recall_tradeoff()

# Example: Different P/R combinations
examples = [
    (0.9, 0.9, "Balanced high"),
    (0.9, 0.5, "High P, Low R"),
    (0.5, 0.9, "Low P, High R"),
    (0.5, 0.5, "Balanced medium")
]

print("\nF1 Score Examples:")
print("="*60)
for p, r, desc in examples:
    f1 = 2 * p * r / (p + r)
    print(f"{desc:20} | P={p:.2f}, R={r:.2f} → F1={f1:.3f}")

# Evaluating Structured Objects

**Q**: How do we evaluate output like syntactic dependency trees, which are structured?

**A**: Break the structures into flat sets of items, and then use precision and recall.

### Example: Dependency Parsing

**True parse**:  
```
                    pmod
              obj─────────────────pobj
    det   subj      det             det
     ↓     ↓         ↓               ↓
    the   cat   chased   the   mouse   from   the   house
     1     2      3       4      5       6      7      8
```

T = {det₂₁, sub₃₂, pmod₃₄, obj₃₅, det₅₄, pobj₆₈, det₈₇}

**Predicted parse**:  
```
                    obj              nmod
                                      mod
    det   subj      det             det
     ↓     ↓         ↓               ↓
    the   cat   chased   the   mouse   from   the   house
     1     2      3       4      5       6      7      8
```

P = {det₂₁, sub₃₂, obj₃₅, det₅₄, det₈₇, nmod₅₈, mod₈₆}

### Metrics:

- \# true positives (P ⋂ T) = 5
- \# false positives (P – T) = 2
- \# false negatives (T – P) = 2

**Precision** = TP / (TP + FP) = 5/7

**Recall** = TP / (TP + FN) = 5/7

In [ ]:
# Example: Evaluating dependency parsing
def evaluate_dependency_parsing(true_deps, pred_deps):
    """Evaluate dependency parsing using sets of (head, dep, label) triples"""
    true_set = set(true_deps)
    pred_set = set(pred_deps)
    
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return tp, fp, fn, precision, recall, f1

# Example from slide
true_deps = [
    (2, 1, 'det'),    # the <- cat
    (3, 2, 'subj'),   # cat <- chased
    (3, 4, 'pmod'),   # chased -> the (prep modifier)
    (3, 5, 'obj'),    # chased -> mouse
    (5, 4, 'det'),    # the <- mouse
    (6, 8, 'pobj'),   # from -> house
    (8, 7, 'det')     # the <- house
]

pred_deps = [
    (2, 1, 'det'),    # the <- cat
    (3, 2, 'subj'),   # cat <- chased
    (3, 5, 'obj'),    # chased -> mouse
    (5, 4, 'det'),    # the <- mouse
    (8, 7, 'det'),    # the <- house
    (5, 8, 'nmod'),   # mouse -> house (different structure)
    (8, 6, 'mod')     # house <- from (different structure)
]

tp, fp, fn, precision, recall, f1 = evaluate_dependency_parsing(true_deps, pred_deps)

print("Dependency Parsing Evaluation")
print("="*50)
print(f"True positives (correct arcs): {tp}")
print(f"False positives (incorrect arcs): {fp}")
print(f"False negatives (missing arcs): {fn}")
print(f"\nPrecision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

print("\nCorrect arcs:", sorted(set(true_deps) & set(pred_deps)))
print("Incorrect arcs:", sorted(set(pred_deps) - set(true_deps)))
print("Missing arcs:", sorted(set(true_deps) - set(pred_deps)))

# Evaluation Against Human Ratings

• What do human ratings look like?

  ### Discrete:
  - In MT, assess **fluency** and **adequacy** using a scale (usually Likert-scale [5- or 7- point])
  - In sentiment classification, label positive/negative/neutral

  ### Continuous: e.g. [0,1]
  - Strongly negative (0) to strongly positive (1) for sentiment polarity of a text

• Can be compiled from crowdsourcing platforms, e.g., Mechanical Turk

# Evaluation Against Human Ratings

• **Question**: How do we evaluate with human ratings, especially when judging automatic metrics?

• **Typical answer**: Correlation between automatic and human assessments

```
   r = 0.7          r = 0.3          r = 0
      •  •            •   •            • •  •
    •   ••          •     •           •   •
  ••    •          •     •            •   •
  •   •            •   •              •  • •
 •  •                •  •               •

   r = -0.7         r = -0.3         r = 0
 •  •                •  •               •
  •   •            •   •              •  • •
  ••    •          •     •            •   •
    •   ••          •     •           •   •
      •  •            •   •            • •  •
```

Image Source: Laerd Statistics - statistics.laerd.com/statistical-guides/pearson-correlation-coefficient-statistical-guide.php, CC BY-SA 4.0, commons.wikimedia.org/w/index.php?curid=84255641

In [ ]:
# Demonstrate correlation analysis
# Generate synthetic data: automatic scores vs human ratings
np.random.seed(42)
human_ratings = np.random.uniform(1, 5, 50)  # Human ratings on 1-5 scale

# Automatic metric with high correlation
auto_metric_high = human_ratings + np.random.normal(0, 0.5, 50)

# Automatic metric with low correlation
auto_metric_low = np.random.uniform(1, 5, 50)

# Calculate correlations
r_high, p_high = pearsonr(human_ratings, auto_metric_high)
r_low, p_low = pearsonr(human_ratings, auto_metric_low)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# High correlation
ax1.scatter(human_ratings, auto_metric_high, alpha=0.6, s=50)
z = np.polyfit(human_ratings, auto_metric_high, 1)
p = np.poly1d(z)
ax1.plot(human_ratings, p(human_ratings), "r--", linewidth=2)
ax1.set_xlabel('Human Ratings', fontsize=12)
ax1.set_ylabel('Automatic Metric (High Correlation)', fontsize=12)
ax1.set_title(f'High Correlation (r = {r_high:.3f}, p < 0.001)', fontsize=14)
ax1.grid(True, alpha=0.3)

# Low correlation
ax2.scatter(human_ratings, auto_metric_low, alpha=0.6, s=50, color='orange')
z2 = np.polyfit(human_ratings, auto_metric_low, 1)
p2 = np.poly1d(z2)
ax2.plot(human_ratings, p2(human_ratings), "r--", linewidth=2)
ax2.set_xlabel('Human Ratings', fontsize=12)
ax2.set_ylabel('Automatic Metric (Low Correlation)', fontsize=12)
ax2.set_title(f'Low Correlation (r = {r_low:.3f}, p = {p_low:.3f})', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Correlation Analysis")
print("="*50)
print(f"High correlation: r = {r_high:.3f}, p-value < 0.001")
print(f"  → Automatic metric explains {r_high**2:.1%} of variance")
print(f"\nLow correlation: r = {r_low:.3f}, p-value = {p_low:.3f}")
print(f"  → Automatic metric explains {r_low**2:.1%} of variance")

# Evaluation Against Human Ratings

### Issues to discuss

• Need for data across the entire range

• Interpretation of r² as % of variance

• Multivariate regression

## Summary & Key Takeaways

### Types of Evaluation
- **Intrinsic**: Direct task performance (accuracy, F1, etc.)
- **Extrinsic**: Performance on downstream tasks
- **Situated**: Real-world deployment scenarios
- **Economic**: Market value and adoption

### Evaluation Methodology
- **Formative**: Iterative development feedback
- **Summative**: Final milestone assessment
- **Shared tasks**: Standardized benchmarks (pros and cons)
- **Leaderboard phenomenon**: SOTA-chasing pitfalls

### Core Metrics
- **Accuracy**: Simple but limited
- **Precision & Recall**: Handle multiple responses
- **F-measure**: Harmonic mean of P&R
- **Inter-annotator agreement**: Upper bound on performance

### Best Practices
- Proper train/dev/test splits
- Cross-validation for robust estimates
- Multiple evaluation metrics
- Comparison to baselines and upper bounds
- Consider both absolute and relative improvements

### Critical Thinking
- Don't just optimize for a single metric
- Consider computational costs
- Assess robustness and generalization
- Understand what the task actually measures

## Next Week Preview

**Topic**: Machine Learning in NLP  
**Focus**: Supervised learning, feature engineering, and classical ML approaches

### Prepare for next week:
- Review basic machine learning concepts
- Think about how to represent text as features
- Consider: What are the strengths and limitations of classical ML vs. deep learning?

---

## Discussion Questions

1. **Evaluation choices**: When would you prefer intrinsic vs. extrinsic evaluation?

2. **Metric selection**: Why might F1 score be preferred over accuracy for imbalanced datasets?

3. **Leaderboards**: What are the pros and cons of competition-based research?

4. **Agreement**: What does low inter-annotator agreement tell us about a task?

5. **Trade-offs**: How do you balance precision vs. recall for different applications?

6. **Future directions**: How might evaluation practices evolve with LLMs and generative AI?